In [ ]:

"""
AI-Powered To-Do List Using Python
------------------------------------
A command-line To-Do List application that goes beyond basic task
management by analyzing each task's urgency (deadline) and importance
(keywords) to automatically suggest a priority score and order.

Core features:
    1. Add task (with optional deadline)
    2. View tasks (sorted by AI-suggested priority)
    3. Mark task as done
    4. Delete task
    5. Exit

Data is stored permanently in a JSON file (tasks.json).
"""

import json
import os
from datetime import datetime, timedelta

FILE_NAME = "tasks.json"

# Keywords used by the simple "AI" model to judge importance.
# In a real ML system these weights could be learned from user
# behaviour; here they act as a lightweight rule-based substitute.
HIGH_IMPORTANCE_WORDS = ["exam", "urgent", "deadline", "submit", "interview", "project"]
MEDIUM_IMPORTANCE_WORDS = ["meeting", "assignment", "report", "call", "class"]

# Keywords used to auto-classify a task into a category.
# This simulates basic NLP-style text classification.
CATEGORY_KEYWORDS = {
    "Study":    ["exam", "assignment", "class", "study", "homework", "project", "submit"],
    "Work":     ["meeting", "report", "client", "office", "deadline", "interview", "call"],
    "Health":   ["gym", "doctor", "workout", "medicine", "appointment", "run"],
    "Personal": ["birthday", "shopping", "family", "friend", "call", "clean"],
}


# ---------------------------------------------------------
# File handling
# ---------------------------------------------------------
def load_tasks():
    """Load tasks from file."""
    if os.path.exists(FILE_NAME):
        with open(FILE_NAME, "r") as f:
            return json.load(f)
    return []


def save_tasks(tasks):
    """Save tasks to file."""
    with open(FILE_NAME, "w") as f:
        json.dump(tasks, f, indent=2)


# ---------------------------------------------------------
# "AI" priority engine
# ---------------------------------------------------------
def calculate_priority_score(task_text, deadline_str):
    """
    Calculates a priority score (higher = more urgent/important).

    Importance is estimated from keywords in the task text.
    Urgency is estimated from how many days remain until the deadline.
    The two scores are combined to give an overall priority.
    """
    score = 0
    text_lower = task_text.lower()

    # --- Importance analysis ---
    if any(word in text_lower for word in HIGH_IMPORTANCE_WORDS):
        score += 50
    elif any(word in text_lower for word in MEDIUM_IMPORTANCE_WORDS):
        score += 25

    # --- Urgency analysis (based on deadline) ---
    if deadline_str:
        try:
            deadline = datetime.strptime(deadline_str, "%Y-%m-%d")
            days_left = (deadline - datetime.now()).days

            if days_left < 0:
                score += 100          # overdue
            elif days_left == 0:
                score += 80           # due today
            elif days_left <= 2:
                score += 60           # due very soon
            elif days_left <= 7:
                score += 30           # due this week
            else:
                score += 10           # due later
        except ValueError:
            pass  # invalid date format, ignore urgency bonus

    return score


def priority_label(score):
    """Convert a numeric score into a readable priority label."""
    if score >= 70:
        return "HIGH"
    elif score >= 35:
        return "MEDIUM"
    else:
        return "LOW"


def classify_category(task_text):
    """
    Auto-classify a task into a category by matching keywords.
    This acts as a simple stand-in for an NLP text-classification model.
    """
    text_lower = task_text.lower()
    best_category = "General"
    best_matches = 0

    for category, keywords in CATEGORY_KEYWORDS.items():
        matches = sum(1 for word in keywords if word in text_lower)
        if matches > best_matches:
            best_matches = matches
            best_category = category

    return best_category


def parse_natural_deadline(deadline_input):
    """
    Understands simple natural-language deadlines like 'today',
    'tomorrow', 'next week', in addition to YYYY-MM-DD.
    This gives the app a small 'understands plain English' feel.
    """
    if not deadline_input:
        return ""

    text = deadline_input.strip().lower()
    today = datetime.now()

    if text == "today":
        return today.strftime("%Y-%m-%d")
    elif text == "tomorrow":
        return (today + timedelta(days=1)).strftime("%Y-%m-%d")
    elif text in ("next week", "nextweek"):
        return (today + timedelta(days=7)).strftime("%Y-%m-%d")

    # Otherwise assume it's already in YYYY-MM-DD format
    return deadline_input.strip()


def suggest_next_task(tasks):
    """
    AI 'assistant' suggestion: looks at all pending tasks and tells
    the user which one to tackle next, based on priority score.
    """
    pending = [t for t in tasks if not t["done"]]
    if not pending:
        print("You're all caught up! No pending tasks.\n")
        return

    top_task = max(pending, key=lambda t: t["priority_score"])
    print(f"\nAI Suggestion: Do this next -> '{top_task['task']}' "
          f"(Category: {top_task.get('category', 'General')}, "
          f"Priority: {top_task['priority']})\n")


# ---------------------------------------------------------
# Task operations
# ---------------------------------------------------------
def add_task(tasks, task_text, deadline_input):
    """Add a new task with AI-calculated priority and category."""
    deadline_str = parse_natural_deadline(deadline_input)
    score = calculate_priority_score(task_text, deadline_str)
    category = classify_category(task_text)

    task = {
        "task": task_text,
        "deadline": deadline_str if deadline_str else "N/A",
        "category": category,
        "priority_score": score,
        "priority": priority_label(score),
        "done": False
    }
    tasks.append(task)
    save_tasks(tasks)
    print(f"Added: '{task_text}'  [Category: {category}, Priority: {task['priority']}]\n")


def view_tasks(tasks):
    """Display all tasks sorted by priority (highest first)."""
    if not tasks:
        print("No tasks yet!\n")
        return

    # Sort a copy so pending tasks with higher priority appear first
    sorted_tasks = sorted(tasks, key=lambda t: (t["done"], -t["priority_score"]))

    print("\n--- Your Tasks (AI-Sorted by Priority) ---")
    for i, item in enumerate(sorted_tasks, 1):
        status = "Done" if item["done"] else "Pending"
        print(f"{i}. [{status}] {item['task']}  "
              f"(Category: {item.get('category', 'General')}, "
              f"Deadline: {item['deadline']}, Priority: {item['priority']})")
    print()


def mark_done(tasks, display_index):
    """Mark a task as done using its position in the sorted view."""
    sorted_tasks = sorted(tasks, key=lambda t: (t["done"], -t["priority_score"]))
    if 0 < display_index <= len(sorted_tasks):
        target = sorted_tasks[display_index - 1]
        for t in tasks:
            if t is target:
                t["done"] = True
        save_tasks(tasks)
        print(f"Marked as done: {target['task']}\n")
    else:
        print("Invalid task number\n")


def delete_task(tasks, display_index):
    """Delete a task using its position in the sorted view."""
    sorted_tasks = sorted(tasks, key=lambda t: (t["done"], -t["priority_score"]))
    if 0 < display_index <= len(sorted_tasks):
        target = sorted_tasks[display_index - 1]
        tasks.remove(target)
        save_tasks(tasks)
        print(f"Deleted: {target['task']}\n")
    else:
        print("Invalid task number\n")


# ---------------------------------------------------------
# Main program
# ---------------------------------------------------------
def main():
    tasks = load_tasks()

    while True:
        print("--- AI-Powered To-Do List ---")
        print("1. Add task")
        print("2. View tasks")
        print("3. Mark task as done")
        print("4. Delete task")
        print("5. AI Suggestion (what to do next)")
        print("6. Exit")

        choice = input("Choose (1-6): ")

        if choice == "1":
            task_text = input("Enter a task: ").strip()
            if not task_text:
                print("Please enter a valid task\n")
                continue
            deadline_str = input(
                "Enter deadline (YYYY-MM-DD, 'today', 'tomorrow', 'next week') or leave blank: "
            ).strip()
            add_task(tasks, task_text, deadline_str)

        elif choice == "2":
            view_tasks(tasks)

        elif choice == "3":
            view_tasks(tasks)
            try:
                index = int(input("Enter task number to mark as done: "))
                mark_done(tasks, index)
            except ValueError:
                print("Please enter a valid number\n")

        elif choice == "4":
            view_tasks(tasks)
            try:
                index = int(input("Enter task number to delete: "))
                delete_task(tasks, index)
            except ValueError:
                print("Please enter a valid number\n")

        elif choice == "5":
            suggest_next_task(tasks)

        elif choice == "6":
            print("Goodbye!")
            break

        else:
            print("Invalid choice, try again\n")


if __name__ == "__main__":
    main()

--- AI-Powered To-Do List ---
1. Add task
2. View tasks
3. Mark task as done
4. Delete task
5. AI Suggestion (what to do next)
6. Exit
Choose (1-6): 1
Enter a task: NPTEL week 12 assignment 
Enter deadline (YYYY-MM-DD, 'today', 'tomorrow', 'next week') or leave blank: 14-10-2026
Added: 'NPTEL week 12 assignment'  [Category: Study, Priority: LOW]

--- AI-Powered To-Do List ---
1. Add task
2. View tasks
3. Mark task as done
4. Delete task
5. AI Suggestion (what to do next)
6. Exit
Choose (1-6): 1
Enter a task: Internal lab exam
Enter deadline (YYYY-MM-DD, 'today', 'tomorrow', 'next week') or leave blank: 2026-10-11
Added: 'Internal lab exam'  [Category: Study, Priority: HIGH]

--- AI-Powered To-Do List ---
1. Add task
2. View tasks
3. Mark task as done
4. Delete task
5. AI Suggestion (what to do next)
6. Exit
Choose (1-6): 1
Enter a task: Blood test 
Enter deadline (YYYY-MM-DD, 'today', 'tomorrow', 'next week') or leave blank: Tomorrow 
Added: 'Blood test'  [Category: General, Priority: 